<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/TRIAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def triad_algorithm(r1, r2, b1, b2):
    """
    Computes the Direction Cosine Matrix (DCM) from the Reference frame
    to the Body frame using the TRIAD algorithm.

    Inputs:
        r1, r2 : Reference vectors (e.g., Sun, Magnetic Field in ECI)
        b1, b2 : Measured vectors (e.g., Sun, Magnetic Field in Body frame)

    Returns:
        C_ref_to_body : 3x3 Direction Cosine Matrix
    """
    # =========================================================================
    # Step 1: Normalize the input vectors
    # =========================================================================
    r1_hat = r1 / np.linalg.norm(r1)
    r2_hat = r2 / np.linalg.norm(r2)
    b1_hat = b1 / np.linalg.norm(b1)
    b2_hat = b2 / np.linalg.norm(b2)

    # =========================================================================
    # Step 2: Construct the Reference Triad (Orthogonal basis in Ref frame)
    # =========================================================================
    # Axis 1: The primary reference vector (assumed to be the most accurate)
    t_r1 = r1_hat

    # Axis 2: Orthogonal to Axis 1, lying in the plane of r1 and r2
    t_r2 = np.cross(r1_hat, r2_hat)
    norm_r2 = np.linalg.norm(t_r2)
    if norm_r2 < 1e-6:
        raise ValueError("Reference vectors r1 and r2 are collinear! TRIAD fails.")
    t_r2 = t_r2 / norm_r2

    # Axis 3: Completes the right-handed orthogonal system
    t_r3 = np.cross(t_r1, t_r2)

    # Form the Reference Triad Matrix (columns are the basis vectors)
    R = np.column_stack((t_r1, t_r2, t_r3))

    # =========================================================================
    # Step 3: Construct the Body Triad (Orthogonal basis in Body frame)
    # =========================================================================
    # Axis 1: The primary body measurement
    t_b1 = b1_hat

    # Axis 2: Orthogonal to Axis 1, lying in the plane of b1 and b2
    t_b2 = np.cross(b1_hat, b2_hat)
    t_b2 = t_b2 / np.linalg.norm(t_b2)

    # Axis 3: Completes the right-handed orthogonal system
    t_b3 = np.cross(t_b1, t_b2)

    # Form the Body Triad Matrix
    B = np.column_stack((t_b1, t_b2, t_b3))

    # =========================================================================
    # Step 4: Solve for the Attitude Matrix (DCM)
    # =========================================================================
    # We want C such that B = C * R.
    # Since R is orthogonal, R^-1 = R^T. Therefore, C = B * R^T.
    C_ref_to_body = B @ R.T

    return C_ref_to_body


In [3]:
def calculate_attitude_error(C_true, C_est):
    """
    Calculates the total angular error (in degrees) between the true
    attitude and the estimated attitude.
    """
    # Relative rotation matrix
    delta_C = C_true @ C_est.T

    # Trace of a rotation matrix is 1 + 2*cos(theta)
    trace = np.trace(delta_C)
    cos_theta = (trace - 1.0) / 2.0

    # Clip to prevent math domain errors from floating point drift
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    error_rad = np.arccos(cos_theta)
    return np.degrees(error_rad)


In [4]:
# =========================================================================
# MAIN EXECUTION: Test Scenario
# =========================================================================
print("--- TRIAD Algorithm Test & Validation ---\n")

# 1. Define a "True" Attitude (30 deg Roll, 20 deg Pitch, 10 deg Yaw)
# We'll build a DCM using standard aerospace 3-2-1 Euler angles
roll, pitch, yaw = np.radians([30.0, 20.0, 10.0])

Rx = np.array([[1, 0, 0], [0, np.cos(roll), np.sin(roll)], [0, -np.sin(roll), np.cos(roll)]])
Ry = np.array([[np.cos(pitch), 0, -np.sin(pitch)], [0, 1, 0], [np.sin(pitch), 0, np.cos(pitch)]])
Rz = np.array([[np.cos(yaw), np.sin(yaw), 0], [-np.sin(yaw), np.cos(yaw), 0], [0, 0, 1]])

C_true = Rz @ Ry @ Rx  # True DCM from Reference to Body

--- TRIAD Algorithm Test & Validation ---



In [5]:
# 2. Define Reference Vectors (in the Reference/ECI frame)
# Let's say the Sun is along the X-axis, and the Magnetic field is along the Y-axis
r_sun = np.array([1.0, 0.0, 0.0])
r_mag = np.array([0.0, 1.0, 0.0])

In [6]:
# 3. Generate "True" Body Measurements by rotating the reference vectors
b_sun_true = C_true @ r_sun
b_mag_true = C_true @ r_mag

In [7]:
# 4. Add Sensor Noise to simulate real hardware
# (e.g., Sun sensor has 0.1 deg noise, Magnetometer has 1.0 deg noise)
np.random.seed(42)
noise_sun = np.random.normal(0, np.radians(0.1), 3)
noise_mag = np.random.normal(0, np.radians(1.0), 3)

b_sun_meas = b_sun_true + noise_sun
b_mag_meas = b_mag_true + noise_mag

In [8]:
# 5. Run the TRIAD Algorithm
# Note: We pass the Sun vectors first because the Sun sensor is more accurate.
# TRIAD will force the solution to perfectly match the first vector.
C_est = triad_algorithm(r_sun, r_mag, b_sun_meas, b_mag_meas)

In [9]:
# 6. Analyze Results
error_deg = calculate_attitude_error(C_true, C_est)

print(f"True Attitude (Euler): Roll={np.degrees(roll):.1f}°, Pitch={np.degrees(pitch):.1f}°, Yaw={np.degrees(yaw):.1f}°")
print(f"Total Attitude Error:  {error_deg:.4f} degrees\n")

print("Estimated DCM (Reference to Body):")
print(np.round(C_est, 4))

print("\nTrue DCM (Reference to Body):")
print(np.round(C_true, 4))

True Attitude (Euler): Roll=30.0°, Pitch=20.0°, Yaw=10.0°
Total Attitude Error:  0.6278 degrees

Estimated DCM (Reference to Body):
[[ 0.9251  0.3214 -0.2019]
 [-0.1632  0.8171  0.5529]
 [ 0.3427 -0.4786  0.8084]]

True DCM (Reference to Body):
[[ 0.9254  0.3188 -0.2049]
 [-0.1632  0.8232  0.5438]
 [ 0.342  -0.4698  0.8138]]
